# A/B-тест геймификации на образовательной платформе

*Кейс «Оценка результатов A/B-тестирования продуктовой фичи»*

## Цель проекта
Оценить успешность проведенного A/B-теста по внедрению новой механики геймификации — системы достижений за регулярное решение задач — на образовательной платформе. На основе статистического анализа необходимо принять аргументированное бизнес-решение: раскатить новую функцию на всех пользователей, отправить ее на доработку или отказаться от нее.

## Описание проекта
Продуктовая команда EdTech-компании выдвинула гипотезу: если добавить в личный кабинет ученика элементы геймификации, то конверсия в полное прохождение курса (Completion Rate) и метрики удержания (Retention 7 и 14 дня) статистически значимо вырастут.

Для проверки гипотезы был запущен A/B-тест. Пользователи были случайным образом разделены на две группы: группа A — контрольная, со старым интерфейсом; группа B — тестовая, с интерфейсом, в который добавлена геймификация. Эксперимент длился 3 недели.

## Данные

Датасет состоит из двух файлов:

**`ab_test_users_groups.csv`**: пользователи эксперимента.

| Колонка | Описание |
|---|---|
| `user_id` | идентификатор пользователя |
| `group` | группа эксперимента |
| `signup_date` | дата регистрации на платформе |
| `grade` | класс ученика |

**`ab_test_events.csv`**: лог действий пользователей.

| Колонка | Описание |
|---|---|
| `user_id` | идентификатор пользователя |
| `timestamp` | дата и время события |
| `event_type` | тип события |

Типы событий:
- `login` — вход на платформу
- `homework_started` — начало выполнения домашнего задания
- `homework_completed` — домашнее задание выполнено
- `return_visit_d7` — возврат на платформу на 7-й день
- `return_visit_d14` — возврат на платформу на 14-й день

## Гипотезы

**Completion Rate**

**H0:** Completion Rate в группах A и B равны – $CR_A = CR_B$

**H1:** Completion Rate в группах A и B не равны – $CR_A \neq CR_B$

**Retention D7**

**H0:** Retention 7-го дня в группах A и B равны – $R^{D7}_A = R^{D7}_B$

**H1:** Retention 7-го дня в группах A и B не равны – $R^{D7}_A \neq R^{D7}_B$

**Retention D14**

**H0:** Retention 14-го дня в группах A и B равны – $R^{D14}_A = R^{D14}_B$

**H1:** Retention 14-го дня в группах A и B не равны – $R^{D14}_A \neq R^{D14}_B$

## План анализа

1. Подготовка данных
2. Sanity check
3. Расчет метрик (+ Time to Value)
4. EDA
5. Выбор критерия
6. Статистический расчет (+ MDE, bootstrap)
7. Интерпретация результатов (+ сегментация по классам)
8. Вывод

## Импорт библиотек

In [166]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

## 1. Подготовка данных

### 1.1 Загрузка и первичный осмотр данных

In [167]:
events = pd.read_csv("../data/raw/ab_test_events.csv")
users = pd.read_csv("../data/raw/ab_test_users_groups.csv")

In [168]:
events.info()
events.head()

<class 'pandas.DataFrame'>
RangeIndex: 179330 entries, 0 to 179329
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   user_id     179330 non-null  str  
 1   timestamp   179330 non-null  str  
 2   event_type  179330 non-null  str  
dtypes: str(3)
memory usage: 4.1 MB


,user_id,timestamp,event_type
0,U300281,2026-05-04 00:06:44,homework_completed
1,U300733,2026-05-04 00:19:56,homework_completed
2,U304499,2026-05-04 00:22:07,login
3,U304266,2026-05-04 00:23:42,homework_started
4,U304266,2026-05-04 00:23:42,homework_started


In [169]:
users.info()
users.head()

<class 'pandas.DataFrame'>
RangeIndex: 6008 entries, 0 to 6007
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   user_id      6008 non-null   str  
 1   group        6008 non-null   str  
 2   signup_date  6008 non-null   str  
 3   grade        6008 non-null   int64
dtypes: int64(1), str(3)
memory usage: 187.9 KB


,user_id,group,signup_date,grade
0,U300000,B,2026-03-09,5
1,U300001,B,2025-09-07,9
2,U300002,B,2026-01-11,9
3,U300003,A,2025-06-07,5
4,U300004,B,2025-12-25,11


In [170]:
events["timestamp"].min(), events["timestamp"].max()

('2026-05-04 00:06:44', '2026-05-24 23:55:43')

### 1.2 Приведение типов и дубликаты

При осмотре видно, что `timestamp` и `signup_date` имеют строковый тип данных. Для корректной работы с датами приведем их к типу datetime. В `events` заметны две одинаковые строки подряд, поэтому также проверим таблицы на дубликаты.

In [171]:
events["timestamp"] = pd.to_datetime(events["timestamp"])
users["signup_date"] = pd.to_datetime(users["signup_date"])

In [172]:
events.duplicated().sum(), users.duplicated().sum()

(np.int64(2), np.int64(0))

In [173]:
events, users = events.drop_duplicates(), users.drop_duplicates()

### 1.3 Пользователи в обеих группах

Пользователь мог попасть в таблицу дважды с разными данными. Проверим количество уникальных `user_id` и сравним его с числом строк.

In [174]:
events.groupby("user_id").size().describe()

count    6000.000000
mean       29.888000
std        33.994888
min         6.000000
25%        24.000000
50%        28.000000
75%        33.000000
max       842.000000
dtype: float64

In [175]:
users.nunique()

user_id        6000
group             2
signup_date     370
grade             7
dtype: int64

In [176]:
users[users["user_id"].duplicated(keep=False)].sort_values("user_id")

,user_id,group,signup_date,grade
357,U300357,A,2026-03-19,9
6000,U300357,B,2026-03-19,9
460,U300460,B,2025-08-12,6
6001,U300460,A,2025-08-12,6
920,U300920,B,2025-04-09,10
6002,U300920,A,2025-04-09,10
1014,U301014,A,2026-03-28,10
6003,U301014,B,2026-03-28,10
1087,U301087,B,2025-06-27,10
6004,U301087,A,2025-06-27,10


8 пользователей числятся одновременно в обеих группах. Определить, какой интерфейс видел такой пользователь, – невозможно – удаляем целиком из обеих таблиц, – на результаты это не повлияет.

In [177]:
duplicates = users[users.duplicated(subset="user_id", keep=False)]["user_id"].unique()

users = users[~users["user_id"].isin(duplicates)]
events = events[~events["user_id"].isin(duplicates)]

### 1.4 Боты
В прошлом шаге при изучении таблицы `events` мы выяснили – у типичного ученика около 28 событий, у 75% – не больше 33, а максимальное значение – 842 – аномально большое число действий. Найдем всех таких пользователей.

In [178]:
events_per_user = events.groupby("user_id").size().sort_values(ascending=False)
events_per_user.head(20)

user_id
U303729    842
U304167    829
U305378    793
U302840    790
U304266    780
U300573    771
U300733    770
U302379    765
U300281    760
U300158    738
U304499    729
U303438    711
U300628     52
U301601     51
U303387     51
U300180     50
U300042     49
U305026     49
U300388     49
U302161     49
dtype: int64

In [179]:
bots = events_per_user[events_per_user > 500]

users = users[~users["user_id"].isin(bots.index)]
events = events[~events["user_id"].isin(bots.index)]

Обнаружено 12 пользоватлей с аномальным количеством действий (711+); следующий по активности идет пользователь с 52 действиями – разрыв более чем в 10 раз. Удаляем их из обеих таблиц.

### 1.5 Объединение таблиц

Для расчета метрик каждому событию нужны группа и класс пользователя. Объединим `users` и `events` по `user_id`.

In [180]:
events_full = users.merge(events, on="user_id", how="left")
events_full

,user_id,group,signup_date,grade,timestamp,event_type
0,U300000,B,2026-03-09,5,2026-05-04 19:36:00,login
1,U300000,B,2026-03-09,5,2026-05-07 19:35:00,login
2,U300000,B,2026-03-09,5,2026-05-07 19:42:00,homework_started
3,U300000,B,2026-03-09,5,2026-05-08 17:05:00,login
4,U300000,B,2026-03-09,5,2026-05-08 17:11:00,homework_started
...,...,...,...,...,...,...
169828,U305999,A,2025-06-19,11,2026-05-20 19:56:00,homework_started
169829,U305999,A,2025-06-19,11,2026-05-20 20:16:00,homework_completed
169830,U305999,A,2025-06-19,11,2026-05-21 20:47:00,login
169831,U305999,A,2025-06-19,11,2026-05-21 20:53:00,homework_started


## 2. Sanity check

## 2.1 Пересечение групп
Ранее мы уже нашли 8 пользователей, находящихся в обеих группах и удалили их

In [181]:
len(users) == users["user_id"].nunique()

True

### 2.2 Проверка на SRM (Sample Ratio Mismatch)

Пользователи должны были распределиться по группам 50/50. Если фактическое соотношение значимо отличается от ожидаемого, это признак поломки сплита, и результатам эксперимента доверять нельзя. Проверим это критерием хи-квадрат согласия.

- **H0:** соотношение групп соответствует 50/50, разница случайна
- **H1:** соотношение отличается от 50/50

Уровень значимости α = 0.01, используем более строгий порог α = 0.01, чтобы снизить риск.

In [182]:
observed = users.groupby("group").size()
expected = [len(users) / 2, len(users) / 2]

print(observed)
stats.chisquare(observed, expected)

group
A    2992
B    2988
dtype: int64


Power_divergenceResult(statistic=np.float64(0.0026755852842809363), pvalue=np.float64(0.9587469721447145))

В группе A 2992 пользователя, в группе B – 2988. p-value ≈ 0.959: если бы сплит действительно был 50/50, то в ~96% случаев мы бы случайно получили такой же перекос или больше. Это намного выше уровня значимости, поэтому оснований отвергнуть H0 нет. SRM не обнаружен, сплит корректен.

### 2.3 Баланс групп по классам

При корректной рандомизации группы должны быть похожи по характеристикам, на которые фича повлиять не могла. Проверим распределение учеников по классам, – если в одной из групп окажется заметно больше учеников определенного класса, сравнение групп будет нечестным.

Используем критерий хи-квадрат независимости.

- **H0:** распределение учеников по классам одинаково в обеих группах
- **H1:** распределение по классам в группах различается

Уровень значимости α = 0.05.

In [183]:
grade_table = pd.crosstab(users["grade"], users["group"])
print(grade_table)

chi2, p_value, dof, expected = stats.chi2_contingency(grade_table)
p_value

group    A    B
grade          
5      446  416
6      446  462
7      402  435
8      454  431
9      430  379
10     409  438
11     405  427


np.float64(0.23723332342252845)

В каждом классе от 379 до 462 учеников на группу, без явного перекоса. p-value ≈ 0.237: если бы распределение по классам в группах действительно было одинаковым, такое или большее расхождение мы бы получали случайно примерно в 24% случаев. Это выше уровня значимости 0.05, поэтому оснований отвергнуть H0 нет. Распределение по классам в группах A и B сопоставимо, рандомизация по этому признаку корректна.


Результатам эксперимента можно доверять, переходим к расчету метрик.

## 3. Расчет метрик

### 3.1 Определения метрик

В данных нет события «курс пройден», поэтому CR определим сами.

| Метрика | Тип | Определение |
|---|---|---|
| Completion Rate | 0/1 | выполнено не менее 3 ДЗ за эксперимент |
| Retention D7 | 0/1 | есть событие `return_visit_d7` |
| Retention D14 | 0/1 | есть событие `return_visit_d14` |
| Доля решенных задач | 0–1 | выполненные ДЗ / начатые ДЗ |

Порог в 3 ДЗ выбран по смыслу – одно задание в неделю до сравнения групп. Устойчивость вывода к выбору порога проверим позже.

In [184]:
events_full[events_full["event_type"] == "homework_completed"].groupby("user_id").size().describe()

count    5874.000000
mean        4.421348
std         2.263584
min         1.000000
25%         3.000000
50%         4.000000
75%         6.000000
max        15.000000
dtype: float64

### 3.2 Метрики на уровне пользователя

In [188]:
user_metrics = events_full.pivot_table(
    index=["user_id", "group", "grade"],  
    columns="event_type",                 
    values="timestamp",                   
    aggfunc="count",                      
    fill_value=0                          
).reset_index()

user_metrics["completion"] = (user_metrics["homework_completed"] >= 3).astype(int)
user_metrics["retention_d7"] = (user_metrics["return_visit_d7"] > 0).astype(int)
user_metrics["retention_d14"]= (user_metrics["return_visit_d14"] > 0).astype(int)
user_metrics["solved_share"] = user_metrics["homework_completed"]/ user_metrics["homework_started"]

metrics = ["completion", "retention_d7", "retention_d14", "solved_share"]

user_metrics[metrics].describe()

event_type,completion,retention_d7,retention_d14,solved_share
count,5980.000000,5980.000000,5980.000000,5980.000000
mean,0.778930,0.753344,0.325251,0.391986
std,0.415002,0.431101,0.468508,0.175064
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000,0.272727
50%,1.000000,1.000000,0.000000,0.384615
75%,1.000000,1.000000,1.000000,0.500000
max,1.000000,1.000000,1.000000,1.000000


### 3.3 Сравнение групп

In [212]:
group_means = user_metrics.groupby("group")[metrics].mean()

delta = group_means.loc["B"] - group_means.loc["A"]
uplift = delta / group_means.loc["A"]

summary = group_means.T
summary["delta_pp"] = delta * 100
summary["uplift_%"] = uplift * 100
summary.round(4)

group,A,B,delta_pp,uplift_%
event_type,,,,
completion,0.7206,0.8373,11.6761,16.2036
retention_d7,0.7269,0.7798,5.2847,7.2698
retention_d14,0.2881,0.3624,7.4348,25.8062
solved_share,0.3461,0.4380,9.1901,26.5560




По всем четырем метрикам группа B выше группы A, но это пока описательные различия:
- Completion Rate: 72.1% и 83.7% (+11.7 п.п., +16.2%)
- Retention D7: 72.7% и 78.0% (+5.3 п.п., +7.3%)
- Retention D14: 28.8% и 36.2% (+7.4 п.п., +25.8%)
- Доля решенных задач: 34.6% и 43.8% (+9.2 п.п., +26.6%)



## 4. EDA